In [3]:
from scenario_gen import generate_scenario

try:
    robots, task_pos, task_payload, task_service, probs, error = generate_scenario(
        n_robots=10,
        n_tasks=50,
        seed=42,
        return_meta=True
    )

    if error:
        print("Could not create a valid scenario:", error)
    else:
        print("OK")
except Exception as e:
    print("Unexpected error:", e)

OK


In [5]:
import numpy as np
import torch
from scenario_gen import generate_scenario, DEPOTS
from drone_energy import DroneEnergyModel

robots, task_pos, task_payload, task_service, _ = generate_scenario(10, 30, seed=65)
model = DroneEnergyModel(robots)
n_robots = len(robots["category"])

# ---- 1. Assign tasks to robots that can carry them ----------------------
remaining = robots["payload_kg"].copy()
assigned = {r: [] for r in range(n_robots)}
unassigned = []

for t in np.argsort(-task_payload):
    fits = np.where(remaining >= task_payload[t])[0]
    if len(fits) == 0:
        unassigned.append(int(t))
        continue
    r = fits[np.argmax(remaining[fits])]
    assigned[r].append(int(t))
    remaining[r] -= task_payload[t]

# ---- 2. Capacity report -------------------------------------------------
print("=== Left-out tasks ===")
if unassigned:
    for t in sorted(unassigned):
        print(f"  T{t}: {task_payload[t]:.2f} kg  (no robot has enough spare capacity)")
    print(f"  Total left out: {len(unassigned)} tasks, "
          f"{task_payload[unassigned].sum():.2f} kg")
else:
    print("  None, all tasks assigned.")

print("\n=== Capacity per drone ===")
for r in range(n_robots):
    cap = robots["payload_kg"][r]
    used = cap - remaining[r]
    if remaining[r] < 1e-6:
        status = "FULL (spare = 0)"
    elif used < 1e-6:
        status = "EMPTY (nothing assigned)"
    else:
        status = "some capacity left"
    print(f"  R{r} (cat {robots['category'][r]}): capacity={cap:.1f} kg, "
          f"used={used:.2f} kg ({used / cap:.0%}), "
          f"spare={remaining[r]:.2f} kg -> {status}")

print(f"\n  Fleet capacity : {robots['payload_kg'].sum():.1f} kg")
print(f"  Assigned load  : {robots['payload_kg'].sum() - remaining.sum():.2f} kg")
print(f"  Spare capacity : {remaining.sum():.2f} kg")

# ---- 3. Order each robot's tasks (nearest neighbour) and evaluate -------
print("\n=== Energy diagnostics ===")
f32 = lambda x: torch.as_tensor(np.asarray(x), dtype=torch.float32)

for r in range(n_robots):
    todo = list(assigned[r])
    if not todo:
        print(f"R{r}: no tasks")
        continue

    depot = DEPOTS[robots["depot_id"][r]]
    order, cur = [], depot
    while todo:
        j = min(todo, key=lambda k: np.linalg.norm(task_pos[k] - cur))
        order.append(j)
        todo.remove(j)
        cur = task_pos[j]

    coords = np.vstack([depot, task_pos[order], depot])
    out = model.tour_energy(
        torch.tensor([r]),
        f32(coords)[None],
        f32(task_payload[order])[None],
        f32(task_service[order])[None],
    )

    load = task_payload[order].sum()
    print(f"R{r} (cat {robots['category'][r]}): tasks={order}")
    print(f"    load={load:.1f}/{robots['payload_kg'][r]:.0f} kg, "
          f"E={out['energy_wh'].item():.1f} Wh, "
          f"SOC drop={out['soc_drop'].item():.1f} %, "
          f"T={out['time_s'].item():.0f} s, "
          f"battery_ok={out['battery_ok'].item()}, "
          f"payload_ok={out['payload_ok'].item()}")

=== Left-out tasks ===
  T0: 2.00 kg  (no robot has enough spare capacity)
  T5: 1.69 kg  (no robot has enough spare capacity)
  T18: 1.68 kg  (no robot has enough spare capacity)
  Total left out: 3 tasks, 5.38 kg

=== Capacity per drone ===
  R0 (cat 1): capacity=10.0 kg, used=8.77 kg (88%), spare=1.23 kg -> some capacity left
  R1 (cat 0): capacity=5.0 kg, used=3.55 kg (71%), spare=1.45 kg -> some capacity left
  R2 (cat 3): capacity=20.0 kg, used=19.70 kg (99%), spare=0.30 kg -> some capacity left
  R3 (cat 1): capacity=10.0 kg, used=8.81 kg (88%), spare=1.19 kg -> some capacity left
  R4 (cat 1): capacity=10.0 kg, used=8.83 kg (88%), spare=1.17 kg -> some capacity left
  R5 (cat 3): capacity=20.0 kg, used=18.63 kg (93%), spare=1.37 kg -> some capacity left
  R6 (cat 2): capacity=15.0 kg, used=13.98 kg (93%), spare=1.02 kg -> some capacity left
  R7 (cat 1): capacity=10.0 kg, used=8.50 kg (85%), spare=1.50 kg -> some capacity left
  R8 (cat 3): capacity=20.0 kg, used=18.60 kg (93%)